# Q.ANT's NPU runs a pretrained ResNet-18 (Image classification)

In [ ]:
# For pretrained weights of the ResNet-18
import torchvision

# For preparation of the image
import torchvision.transforms as transforms
from torch import topk
from PIL import Image
import requests
import matplotlib.pyplot as plt

from qant_resnet import QantResNet18, to_class_prob
import utils

# Read the category labels
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

### Prepare input and neural network data

1. Load pretrained ResNet of PyTorch repository

In [ ]:
w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1.get_state_dict()
qant_model = QantResNet18(w)


2. Load a image from the web

In [ ]:
# select a image by commenting out

url = "https://files.qant.com/s/A2mJmLBa9TMWJLe/download/pezibear-maltese.jpg"  # dog (maltese)
# url = "https://files.qant.com/s/ExCZsBzG2SNoG7B/download/rabbit.jpg"  # rabbit (hare)
# url = "https://files.qant.com/s/aiwxLKjPMeeLmfc/download/train.jpg" # train


# url = "" # custom url

image = Image.open(requests.get(url, stream=True).raw)

preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
input_tensor = preprocess(image)
input_batch = input_tensor.unsqueeze(0)  # create a mini-batch as expected by the model

_ = plt.imshow(image)

## Neural network inference

In [ ]:
# The NN inference
output = qant_model(input_batch)

# Converting output features to probabilites (e.g., Softmax)
probabilities = to_class_prob(output)

# Show top categories
top5_prob, top5_id = topk(probabilities, 5)
top5_label = [categories[id] for id in top5_id]
top5_prob_list = top5_prob.tolist()
utils.display_result(top5_label, top5_prob)